In [1]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:
#augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.15),  # ±15% rotation
    tf.keras.layers.RandomZoom(0.1),       # zoom in/out ±10%
    tf.keras.layers.RandomTranslation(0.1, 0.1)  # small shift ~ shear effect
])

In [3]:
## Preprocessig
### image Preprocessig
training_set = tf.keras.utils.image_dataset_from_directory(
    r'train/Farmer',
    labels="inferred",
    label_mode="categorical",
    class_names=None,
    color_mode="rgb",
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False
)

Found 3039 files belonging to 10 classes.


In [4]:
training_set = training_set.map(lambda x, y: (data_augmentation(x, training=True), y))


In [5]:
### Validation
validation_set = tf.keras.utils.image_dataset_from_directory(
    r'valid/Farmer',
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=32,
    image_size=(128, 128),
    shuffle=False,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False
)

Found 653 files belonging to 10 classes.


In [6]:
# Prefetching for performance
AUTOTUNE = tf.data.AUTOTUNE
training_set = training_set.prefetch(buffer_size=AUTOTUNE)
validation_set = validation_set.prefetch(buffer_size=AUTOTUNE)

In [7]:
training_set

<PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10), dtype=tf.float32, name=None))>

In [8]:
for x,y in training_set:
    print(x,x.shape)
    print(y,y.shape)
    break

tf.Tensor(
[[[[102.84604   142.6547     73.25027  ]
   [ 91.545685  130.8893     66.24812  ]
   [ 82.25726   119.77786    59.17425  ]
   ...
   [ 99.38238   128.82962    25.333248 ]
   [ 95.61003   123.55904    25.352175 ]
   [ 87.24107   115.18627    20.388819 ]]

  [[103.75548   143.40208    80.63019  ]
   [ 96.97991   135.44168    74.963394 ]
   [ 90.71678   128.4624     70.02594  ]
   ...
   [ 91.522156  121.1642     21.144464 ]
   [ 86.21287   114.06407    18.696377 ]
   [ 80.62222   107.77399    14.937214 ]]

  [[102.23648   139.25426    83.33516  ]
   [101.57495   135.68083    80.71533  ]
   [ 98.31865   131.14084    77.05228  ]
   ...
   [ 81.35805   107.83029    15.9910965]
   [ 81.28844   104.43058    15.96203  ]
   [ 87.30908   104.022156   16.882309 ]]

  ...

  [[ 18.436634   34.549854    9.653816 ]
   [ 13.523718   27.449833    8.523028 ]
   [ 25.828636   41.540024   14.856277 ]
   ...
   [ 76.41554   119.35312    43.597656 ]
   [ 71.1334    112.82121    46.311302 ]
   [ 

In [9]:
#model Building
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.models import load_model

In [10]:
model = load_model('best_Plantvillage.h5')

In [11]:
for layer in model.layers[:-10]:
    layer.trainable = False

for layer in model.layers[:-10:]:
    layer.trainable = True


In [12]:
from tensorflow.keras.optimizers import Adam

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [13]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 128, 128, 32)      896       
                                                                 
 conv2d_1 (Conv2D)           (None, 126, 126, 32)      9248      
                                                                 
 max_pooling2d (MaxPooling2D  (None, 63, 63, 32)       0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 63, 63, 64)        18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 61, 61, 64)        36928     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 30, 30, 64)       0         
 2D)                                                    

In [14]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# Callbacks
checkpoint = ModelCheckpoint('best_TF.h5', monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

# Training
history = model.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=50,
    callbacks=[checkpoint, reduce_lr],
    shuffle=True
)

Epoch 1/50
95/95 [==============================] - ETA: 0s - loss: 2.2762 - accuracy: 0.2277
Epoch 1: val_loss improved from inf to 1.94792, saving model to best_TF.h5
95/95 [==============================] - 118s 1s/step - loss: 2.2762 - accuracy: 0.2277 - val_loss: 1.9479 - val_accuracy: 0.3109 - lr: 1.0000e-04
Epoch 2/50
95/95 [==============================] - ETA: 0s - loss: 1.8324 - accuracy: 0.3613
Epoch 2: val_loss improved from 1.94792 to 1.84252, saving model to best_TF.h5
95/95 [==============================] - 116s 1s/step - loss: 1.8324 - accuracy: 0.3613 - val_loss: 1.8425 - val_accuracy: 0.3247 - lr: 1.0000e-04
Epoch 3/50
95/95 [==============================] - ETA: 0s - loss: 1.7033 - accuracy: 0.4018
Epoch 3: val_loss improved from 1.84252 to 1.70179, saving model to best_TF.h5
95/95 [==============================] - 114s 1s/step - loss: 1.7033 - accuracy: 0.4018 - val_loss: 1.7018 - val_accuracy: 0.4089 - lr: 1.0000e-04
Epoch 4/50
95/95 [==========================

KeyboardInterrupt: 